In [1]:
import torch
import tifffile
import numpy as np
import os
from tqdm import tqdm
from skimage import exposure
from micro_sam.automatic_segmentation import get_predictor_and_segmenter, automatic_instance_segmentation

2025-12-21 12:35:33.083470: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:477] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1766316933.098850 1196693 cuda_dnn.cc:8310] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1766316933.104033 1196693 cuda_blas.cc:1418] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered


In [2]:
# ============================
# 1. CONFIGURATION
# ============================
MODEL_TYPE = "vit_b_lm"
CHECKPOINT_PATH = r"/work/scratch/rawat/microsam_training_output/checkpoints/best_bouton_model/best.pt"
INPUT_IMAGE_PATH = r"/home/students/rawat/czi_output/Brain2_PN_KC_Group3_25minfix_OvernightSecAB_488_561_Calyx1_Confocal.tif"
OUTPUT_MASK_PATH = r"/work/scratch/rawat/ouput_masks/Control_group_Brain1_mask.tif"
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

print(f"Using device: {DEVICE}")
print(f"Using model: {MODEL_TYPE} with checkpoint: {CHECKPOINT_PATH}")
print(f"Input image: {INPUT_IMAGE_PATH}")

Using device: cuda
Using model: vit_b_lm with checkpoint: /work/scratch/rawat/microsam_training_output/checkpoints/best_bouton_model/best.pt
Input image: /home/students/rawat/czi_output/Brain2_PN_KC_Group3_25minfix_OvernightSecAB_488_561_Calyx1_Confocal.tif


In [3]:
raw_data = tifffile.imread(INPUT_IMAGE_PATH)
print(f"Original shape: {raw_data.shape}")

# Extract axis order from metadata
axis_order = None
with tifffile.TiffFile(INPUT_IMAGE_PATH) as tif:
    print("Available series:", len(tif.series))
    if tif.pages:
        print("Tags:", list(tif.pages[0].tags.keys())[:10])  # First 10 tags
    print("Shape:", tif.series[0].shape if tif.series else "No series")
    if tif.series:
        axis_order = tif.series[0].axes
        print(f"✅ Axis Order from file: {axis_order}")

# Extract Channel 0 based on axis order
if axis_order == 'ZCYX':
    # (Z, C, Y, X) -> extract channel 0
    image_data = raw_data[:, 0, :, :]
    print(f"Extracted from ZCYX format")
elif axis_order == 'CZYX':
    # (C, Z, Y, X) -> extract channel 0
    image_data = raw_data[0, :, :, :]
    print(f"Extracted from CZYX format")
elif axis_order == 'ZYCX':
    # (Z, Y, C, X) -> extract channel 0
    image_data = raw_data[:, :, 0, :]
    print(f"Extracted from ZYCX format")
else:
    # Fallback: try to guess
    print(f"⚠️ Unknown axis order: {axis_order}. Attempting heuristic extraction...")
    if raw_data.ndim == 4:
        if raw_data.shape[0] <= 4:  # (C, Z, Y, X)
            raw_data = np.transpose(raw_data, (1, 2, 3, 0))
        elif raw_data.shape[1] <= 4:  # (Z, C, Y, X)
            raw_data = np.transpose(raw_data, (0, 2, 3, 1))
    image_data = raw_data[..., 0]

print(f"Processed shape (Z, Y, X): {image_data.shape}")

Original shape: (65, 2, 1101, 1101)
Available series: 1
Tags: [254, 256, 257, 258, 259, 262, 270, 273, 277, 278]
Shape: (65, 2, 1101, 1101)
✅ Axis Order from file: ZCYX
Extracted from ZCYX format
Processed shape (Z, Y, X): (65, 1101, 1101)


In [4]:
# ============================
# 3. INITIALIZE MODEL
# ============================
print("Loading model...")
# We load the model ONCE, then reuse it for every slice
predictor, segmenter = get_predictor_and_segmenter(
    model_type=MODEL_TYPE,
    checkpoint=CHECKPOINT_PATH,
    device=DEVICE,
)

# ============================
# 4. RUN INFERENCE LOOP (One by One)
# ============================
prediction_stack = []

print(f"Starting Slice-by-Slice Inference on {image_data.shape[0]} slices...")

# Tqdm gives you a progress bar
for z in tqdm(range(image_data.shape[0]), desc="Processing Slices"):
    
    # A. Get single slice
    img_slice = image_data[z] # (Y, X)
    
    # B. Skip empty slices (Optional speedup)
    if img_slice.max() == 0:
        prediction_stack.append(np.zeros_like(img_slice))
        continue

    # C. Normalize (Crucial!)
    p2, p99 = np.percentile(img_slice, (2, 99))
    img_slice = exposure.rescale_intensity(
        img_slice, in_range=(p2, p99), out_range=(0, 255)
    ).astype(np.uint8)

    # D. Make 3-Channel (RGB)
    img_rgb = np.stack((img_slice,)*3, axis=-1)

    # E. Predict (ndim=2 because it's just one image)
    # We pass the pre-loaded 'predictor' so it's fast
    pred_2d = automatic_instance_segmentation(
        predictor=predictor,
        segmenter=segmenter,
        input_path=img_rgb,
        ndim=2, # <--- 2D MODE
        # tile_shape=(512, 512), # Optional: tiling within the 2D slice
        halo=(64, 64)
    )
    
    prediction_stack.append(pred_2d)

# ============================
# 5. SAVE RESULT
# ============================
print("Stacking results...")
final_stack = np.array(prediction_stack)

print(f"Saving 3D result to: {OUTPUT_MASK_PATH}")
tifffile.imwrite(OUTPUT_MASK_PATH, final_stack)
print("Done!")

Loading model...
Starting Slice-by-Slice Inference on 65 slices...


Processing Slices: 100%|██████████| 65/65 [00:33<00:00,  1.95it/s]


Stacking results...
Saving 3D result to: /work/scratch/rawat/ouput_masks/Control_group_Brain1_mask.tif
Done!
